In [1]:
import pyspark, os, sys, shutil, pandas as pd
from datetime import datetime
from pathlib import Path
from urllib.parse import urlparse
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T
os.environ['PYSPARK_PYTHON'] = os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable
print("Python", sys.executable)
print("Java", os.environ["JAVA_HOME"])

PROJECT_DIR   = Path(os.getcwd())
WAREHOUSE_DIR = PROJECT_DIR / "spark-warehouse"   # Spark decides layout here  -> MANAGED tables
EXTERNAL_DIR  = PROJECT_DIR / "external-data"     # storage that WE control     -> EXTERNAL tables
LANDING_DIR   = PROJECT_DIR / "landing"           # raw files as delivered by the source system

METASTORE_DB = PROJECT_DIR / "metastore_db"

spark = (SparkSession.builder
         .appName("semantic-layering-and-meta-data-preparation")
         .config("spark.sql.warehouse.dir", WAREHOUSE_DIR.as_posix())
         .config("spark.sql.warehouse.dir", "/path/to/course/spark-warehouse")
         .config("spark.hadoop.javax.jdo.option.ConnectionURL",
                 f"jdbc:derby:;databaseName={METASTORE_DB};create=true")
         .enableHiveSupport()          # <- persistent metastore (embedded Derby in ./metastore_db)
         .getOrCreate())
sc = spark.sparkContext
sc.setLogLevel("WARN")

print("Spark version          :", spark.version)
print("Catalog implementation :", spark.conf.get("spark.sql.catalogImplementation"))   # expect: hive
print("Warehouse directory    :", spark.conf.get("spark.sql.warehouse.dir"))
spark

Python /usr/local/bin/python3.12
Java /usr/lib/jvm/java-21-openjdk-arm64


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/usr/local/lib/python3.12/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/11/10 15:01:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark version          : 4.3.0
Catalog implementation : hive
Warehouse directory    : file:/path/to/course/spark-warehouse


In [2]:
spark.catalog.listTables('lab')

[Table(name='sales_external', catalog='spark_catalog', namespace=['lab'], description=None, tableType='EXTERNAL', isTemporary=False),
 Table(name='sales_managed', catalog='spark_catalog', namespace=['lab'], description=None, tableType='MANAGED', isTemporary=False)]

In [4]:
spark.sql("select * from lab.sales_external limit 10").show()

+---------+---------+--------------------+--------+----------------+---------+----------+--------------+-------------+
|InvoiceNo|StockCode|         Description|Quantity|     InvoiceDate|UnitPrice|CustomerID|       Country|invoice_month|
+---------+---------+--------------------+--------+----------------+---------+----------+--------------+-------------+
|   574740|    22027|TEA PARTY BIRTHDA...|      24| 11/6/2011 16:07|     0.42|   12357.0|   Switzerland|      2011-11|
|   579525|    21977|PACK OF 60 PINK P...|      48| 11/30/2011 8:23|     0.55|   15113.0|United Kingdom|      2011-11|
|   575739|    23243|SET OF TEA COFFEE...|       1| 11/11/2011 9:05|    10.79|      NULL|United Kingdom|      2011-11|
|   576274|    23103|JINGLE BELL HEART...|       1|11/14/2011 14:24|     1.65|   17793.0|United Kingdom|      2011-11|
|   577337|    23313|VINTAGE CHRISTMAS...|       2|11/18/2011 14:51|     4.95|   15494.0|United Kingdom|      2011-11|
|   578344|    23084|  RABBIT NIGHT LIGHT|      

In [5]:
spark.sql("select * from lab.sales_managed limit 10").show()

+---------+---------+--------------------+--------+----------------+---------+----------+--------------+-------------+
|InvoiceNo|StockCode|         Description|Quantity|     InvoiceDate|UnitPrice|CustomerID|       Country|invoice_month|
+---------+---------+--------------------+--------+----------------+---------+----------+--------------+-------------+
|   574740|    22027|TEA PARTY BIRTHDA...|      24| 11/6/2011 16:07|     0.42|   12357.0|   Switzerland|      2011-11|
|   579525|    21977|PACK OF 60 PINK P...|      48| 11/30/2011 8:23|     0.55|   15113.0|United Kingdom|      2011-11|
|   575739|    23243|SET OF TEA COFFEE...|       1| 11/11/2011 9:05|    10.79|      NULL|United Kingdom|      2011-11|
|   576274|    23103|JINGLE BELL HEART...|       1|11/14/2011 14:24|     1.65|   17793.0|United Kingdom|      2011-11|
|   577337|    23313|VINTAGE CHRISTMAS...|       2|11/18/2011 14:51|     4.95|   15494.0|United Kingdom|      2011-11|
|   578344|    23084|  RABBIT NIGHT LIGHT|      

In [6]:
spark.stop()